In [ ]:
# @title Set ONLINE=False if running locally
ONLINE = True

In [ ]:
# @title Imports
import pandas as pd
import numpy as np
import math
import gdown
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

In [ ]:
# Download training and submission data
if ONLINE:
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

In [ ]:
# load training data
number_of_users, number_of_movies = (10000, 1000)
data_pd = pd.read_csv('data_train.csv')

# Split the dataset into train and test
train_size = 0.9
train_pd, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

In [ ]:
def extract_users_items_predictions(data_pd):
    users, movies = \
        [np.squeeze(arr) for arr in np.split(data_pd.Id.str.extract('r(\d+)_c(\d+)').values.astype(int) - 1, 2, axis=-1)]
    predictions = data_pd.Prediction.values
    return users, movies, predictions

In [ ]:
# extract matrix data
train_users, train_movies, train_predictions = extract_users_items_predictions(train_pd)
test_users, test_movies, test_predictions = extract_users_items_predictions(test_pd)

In [ ]:
# create full matrix of observed values
data = np.full((number_of_users, number_of_movies), 0)
mask = np.full((number_of_users, number_of_movies), 0) # 0 -> unobserved value, 1->observed value

for user, movie, pred in zip(train_users, train_movies, train_predictions):
    data[user][movie] = pred
    mask[user][movie] = 1

In [ ]:
# define RMSE metric
rmse = lambda x, y: math.sqrt(mean_squared_error(x, y))

def get_score(predictions, target_values=test_predictions):
    return rmse(predictions, target_values)

def extract_prediction_from_full_matrix(reconstructed_matrix, users=test_users, movies=test_movies):
    # returns predictions for the users-movies combinations specified based on a full m \times n matrix
    assert(len(users) == len(movies)), "users-movies combinations specified should have equal length"
    predictions = np.zeros(len(users))

    for i, (user, movie) in enumerate(zip(users, movies)):
        predictions[i] = reconstructed_matrix[user][movie]

    return predictions

In [ ]:
# @title Use GPU in colab: Runtime->Change Runtime type

#  use gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Using device:', device)

In [ ]:
# @title Model definition
class Encoder(nn.Module):
    def __init__(self, input_dimension, hidden_dimension=128, encoded_dimension=64, drop_p=0.1, hidden_layers=0, activation=nn.SELU()):
        super().__init__()

        modules = []

        # input layer
        modules.append(nn.Linear(in_features=input_dimension, out_features=hidden_dimension))
        modules.append(activation)

        # hidden layers
        c = hidden_dimension - encoded_dimension
        for i in range(hidden_layers):
            modules.append(nn.Linear(in_features=encoded_dimension + c//(2**i), out_features=encoded_dimension + c//(2**(i+1))))
            modules.append(activation)

        # output layer
        modules.append(nn.Linear(in_features=encoded_dimension + c//(2**hidden_layers), out_features=encoded_dimension))
        modules.append(activation)

        # dropout layer
        modules.append(nn.Dropout(drop_p))

        self.model = nn.Sequential(*modules)

    def forward(self, data):
        return self.model(data)

class Decoder(nn.Module):
    def __init__(self, output_dimension, hidden_dimension=128, encoded_dimension=16, drop_p=0.1, hidden_layers=1, activation=nn.SELU()):
        super().__init__()

        modules = []

        c = hidden_dimension - encoded_dimension
        # input layer
        modules.append(nn.Linear(in_features=encoded_dimension, out_features=encoded_dimension + c//(2**hidden_layers)))
        modules.append(activation)

        # hidden layers
        for i in range(hidden_layers, 0, -1):
            modules.append(nn.Linear(in_features=encoded_dimension + c//(2**i), out_features=encoded_dimension + c//(2**(i-1))))
            modules.append(activation)

        # output layer
        modules.append(nn.Linear(in_features=hidden_dimension, out_features=output_dimension))
        modules.append(activation)

        self.model = nn.Sequential(*modules)

    def forward(self, encodings):
        return self.model(encodings)


class AutoEncoder(nn.Module):
    def __init__(self, encoder, decoder, main_dimension, second_dimension, normalize=True, use_bias_1=True, use_bias_2=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.normalize = normalize
        self.use_bias_1 = use_bias_1
        self.use_bias_2 = use_bias_2
        self.main_dimension = main_dimension
        self.second_dimension = second_dimension

        if self.use_bias_1:
            self.bias_1 = torch.nn.Parameter(torch.ones((self.second_dimension, 1)))
        if self.use_bias_2:
            self.bias_2 = torch.nn.Parameter(torch.ones((1, self.main_dimension)))

    def forward(self, data, ids):
        if self.normalize:
            data = torch.nn.functional.normalize(data)
        z = self.encoder(data)
        data_hat = self.decoder(z)
        if self.use_bias_1:
            data_hat += self.bias_1[ids].repeat(1, self.main_dimension)
        if self.use_bias_2:
            data_hat += self.bias_2.repeat(data_hat.shape[0], 1)

        return data_hat

In [ ]:
# @title Hyperparameters
batch_size = 1024
num_epochs = 400
show_validation_score_every_epochs = 1
encoded_dimension = 128
hidden_dimension = 512
encode_users = True
main_dimension = number_of_movies if encode_users else number_of_users
second_dimension = number_of_users if encode_users else number_of_movies
drop_p = 0.1
activation = nn.SELU()
hidden_layers_encoder = 1
hidden_layers_decoder = 1
learning_rate = 0.001
weight_decay = 0.0002
patience=20
normalize = False
use_bias_1 = True
use_bias_2 = True

In [ ]:
# @title Model instantiation
autoencoder = AutoEncoder(
    encoder=Encoder(
        input_dimension=main_dimension,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_encoder,
        activation=activation,
    ),
    decoder=Decoder(
        output_dimension=main_dimension,
        hidden_dimension=hidden_dimension,
        encoded_dimension=encoded_dimension,
        drop_p=drop_p,
        hidden_layers=hidden_layers_decoder,
        activation=activation,
    ),
    main_dimension=main_dimension,
    second_dimension=second_dimension,
    normalize=normalize,
    use_bias_1=use_bias_1,
    use_bias_2=use_bias_2,
).to(device)

In [ ]:
# @title Dataloaders
data_torch = torch.tensor(data, device=device).float()
mask_torch = torch.tensor(mask, device=device)
id_torch = torch.tensor(list(range(second_dimension)), device=device)

if not encode_users:
    data_torch = data_torch.T
    mask_torch = mask_torch.T

dataloader = DataLoader(
    TensorDataset(data_torch, mask_torch, id_torch),
    batch_size=batch_size
)

In [ ]:
# @title optimizer, scheduler, loss function
optimizer = optim.Adam(
    autoencoder.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    'min',
    patience=28,
    factor=0.5,
)

# L2 loss between original ratings and reconstructed ratings for the observed values
def loss_function(original, reconstructed, mask):
    return torch.mean(mask * (original - reconstructed) ** 2)

# reconstuct the whole array
def reconstruct_whole_matrix(autoencoder):

    with torch.no_grad():
        data_reconstructed = autoencoder(data_torch, id_torch).detach().cpu().numpy().clip(1,5)
        if not encode_users:
            data_reconstructed = data_reconstructed.T

    return data_reconstructed

In [ ]:
# @title Training
from copy import deepcopy

best_error = math.inf
best_model_state = None

step = 0
with tqdm(total=len(dataloader) * num_epochs) as pbar:
    for epoch in range(num_epochs):
        for data_batch, mask_batch, id_batch in dataloader:
            optimizer.zero_grad()

            reconstructed_batch = autoencoder(data_batch, id_batch)

            loss = loss_function(data_batch, reconstructed_batch, mask_batch)

            loss.backward()

            optimizer.step()

            pbar.update(1)
            step += 1

        if epoch % show_validation_score_every_epochs == 0:
            autoencoder.eval()
            reconstructed_matrix = reconstruct_whole_matrix(autoencoder)
            predictions = extract_prediction_from_full_matrix(reconstructed_matrix)
            reconstruction_rmse = get_score(predictions)
            if reconstruction_rmse < best_error:
                best_error = reconstruction_rmse
                best_model_state = deepcopy(autoencoder.state_dict())
            pbar.set_description('At epoch {:3d} loss is {:.4f}, best loss {:.4f}'.format(epoch + 1, reconstruction_rmse, best_error))
            autoencoder.train()
        scheduler.step(reconstruction_rmse)

In [ ]:
# @title Save model and generate predictions for ensemble
predictions_path = 'predictions'

if not os.path.exists(predictions_path):
    os.mkdir(predictions_path)

autoencoder.load_state_dict(best_model_state)
autoencoder.eval()

submission_pd = pd.read_csv('sampleSubmission.csv')
challenge_users, challenge_movies, _ = extract_users_items_predictions(submission_pd)

reconstructed_matrix = reconstruct_whole_matrix(autoencoder)
challenge_predictions = extract_prediction_from_full_matrix(reconstructed_matrix, users=challenge_users, movies=challenge_movies).clip(1,5)

test_predictions = extract_prediction_from_full_matrix(reconstructed_matrix).clip(1,5)

np.savez_compressed(os.path.join(predictions_path, 'ae_predictions'), test=test_predictions,
                        challenge=challenge_predictions)

In [ ]:
# @title Save challenge prediction to csv
csv_path = 'csvs'
if not os.path.exists(csv_path):
    os.mkdir(csv_path)

submission_pd['Prediction'] = challenge_predictions

# Save the predictions to a new CSV file
submission_pd.to_csv(os.path.join(csv_path, 'submission_ae.csv'), index=False)